# POS/CASH Balance Feature Engineering

Here we turn monthly POS/CASH records into **one row for every applicant in the
previous-application mapping**. We first summarize each `SK_ID_PREV`, then group
those contracts by `SK_ID_CURR`.

هنجمع الشهور لكل عقد الأول، وبعد كده نجمع العقود لكل عميل.

We follow the small cells, named Pandas aggregations, English/Arabic explanations,
and Parquet convention in `01_bureau.ipynb` and `02_previous_application.ipynb`.
The POS/CASH decisions come from `../eda/04_pos_cash_eda.ipynb` and are checked again
against the CSV. We do not load an application outcome or use `TARGET`.


## 1. Imports and Paths

`Path` builds paths without depending on the operating system. We search the
current directory and its parents, so this notebook runs from the project root
or its own directory. `DATA_RAW` and `DATA_INTERIM` follow the Bureau naming.


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import hashlib
import gc
from IPython.display import display, Markdown


In [4]:
# Locate the project without a personal username, drive, or fixed notebook depth.
REPO_ROOT = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "data/raw/home-credit-default-risk").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook inside the project and extract the Kaggle CSVs "
        "into data/raw/home-credit-default-risk/."
    )
DATA_RAW = REPO_ROOT / "data/raw/home-credit-default-risk"
DATA_INTERIM = REPO_ROOT / "data/interim"
DATA_PROCESSED = REPO_ROOT / "data/processed"
POS_PATH = DATA_RAW / "POS_CASH_balance.csv"
PREV_PATH = DATA_RAW / "previous_application.csv"
OUTPUT_PATH = DATA_INTERIM / "pos_cash_features.parquet"
print("Project root:", REPO_ROOT)


Project root: e:\Depi Notes Data Engineering\DEPI project R5\ProJect


A SHA-256 fingerprint checks file contents, rather than just file size or date.
We keep fingerprints for both raw inputs and compare them at the end.


In [5]:
raw_sha256_before = {}
for path in [POS_PATH, PREV_PATH]:
    with path.open("rb") as raw_file:
        raw_sha256_before[path] = hashlib.file_digest(raw_file, "sha256").hexdigest()


## 2. Load the Required Datasets

The EDA reported about ten million monthly rows. Compact IDs, months, and repeated
status categories reduce memory. Nullable `Int32` DPD columns preserve missing
days instead of converting them into zero. Installment counts use `float32`;
we verify that their observed values are small whole numbers, which this type
represents exactly. Means calculated from these columns are checked with an
appropriate floating-point tolerance.

We read **only the two mapping IDs** from `previous_application.csv`.
The actual POS CSV contains `SK_ID_CURR`; we inspect and audit it, then remove
it. Applicant feature IDs will always come from the previous-application mapping.
The workflow also works when that optional source applicant column is absent.


In [6]:
source_columns = pd.read_csv(POS_PATH, nrows=0).columns.tolist()
required_columns = [
    "SK_ID_PREV", "MONTHS_BALANCE", "NAME_CONTRACT_STATUS",
    "SK_DPD", "SK_DPD_DEF", "CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE"
]
assert set(required_columns).issubset(source_columns)
has_source_applicant_id = "SK_ID_CURR" in source_columns
POS_DTYPES = {
    "SK_ID_PREV": "int32", "SK_ID_CURR": "int32", "MONTHS_BALANCE": "int16",
    "NAME_CONTRACT_STATUS": "category", "SK_DPD": "Int32", "SK_DPD_DEF": "Int32",
    "CNT_INSTALMENT": "float32", "CNT_INSTALMENT_FUTURE": "float32"
}
df = pd.read_csv(
    POS_PATH, usecols=required_columns + (["SK_ID_CURR"] if has_source_applicant_id else []),
    dtype={name: dtype for name, dtype in POS_DTYPES.items() if name in source_columns},
    keep_default_na=False, na_values=[""]
)
previous_mapping = pd.read_csv(
    PREV_PATH, usecols=["SK_ID_PREV", "SK_ID_CURR"],
    dtype={"SK_ID_PREV": "int32", "SK_ID_CURR": "int32"}
)


## 3. Basic Inspection

Rows are **contract × relative month**, not applicants or separate loans.
`MONTHS_BALANCE = -1` is more recent than `-12`. The largest observed offset gives
the latest snapshot. All retained observations must precede the current application.


In [7]:
source_shape = (len(df), len(source_columns))
n_source_contracts = df["SK_ID_PREV"].nunique()
print("Source dimensions:", source_shape)
print("Unique previous contracts:", n_source_contracts)
print("Source has SK_ID_CURR:", has_source_applicant_id)
df.head()


Source dimensions: (10001358, 8)
Unique previous contracts: 936325
Source has SK_ID_CURR: True


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


In [8]:
df.info(memory_usage="deep")
display(df.isna().sum().rename("missing_source_rows").to_frame())
display(df["NAME_CONTRACT_STATUS"].value_counts(dropna=False))
display(df[["MONTHS_BALANCE", "SK_DPD", "SK_DPD_DEF",
            "CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE"]].describe())


<class 'pandas.DataFrame'>
RangeIndex: 10001358 entries, 0 to 10001357
Data columns (total 8 columns):
 #   Column                 Dtype   
---  ------                 -----   
 0   SK_ID_PREV             int32   
 1   SK_ID_CURR             int32   
 2   MONTHS_BALANCE         int16   
 3   CNT_INSTALMENT         float32 
 4   CNT_INSTALMENT_FUTURE  float32 
 5   NAME_CONTRACT_STATUS   category
 6   SK_DPD                 Int32   
 7   SK_DPD_DEF             Int32   
dtypes: Int32(2), category(1), float32(2), int16(1), int32(2)
memory usage: 276.6 MB


,missing_source_rows
SK_ID_PREV,0
SK_ID_CURR,0
MONTHS_BALANCE,0
CNT_INSTALMENT,26071
CNT_INSTALMENT_FUTURE,26087
NAME_CONTRACT_STATUS,0
SK_DPD,0
SK_DPD_DEF,0


NAME_CONTRACT_STATUS
Active                   9151119
Completed                 744883
Signed                     87260
Demand                      7065
Returned to the store       5461
Approved                    4917
Amortized debt               636
Canceled                      15
XNA                            2
Name: count, dtype: int64

,MONTHS_BALANCE,SK_DPD,SK_DPD_DEF,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE
count,1.000136e+07,10001358.0,10001358.0,9.975287e+06,9.975271e+06
mean,-3.501259e+01,11.606928,0.654468,1.708965e+01,1.048384e+01
std,2.606657e+01,132.714043,32.762491,1.199506e+01,1.110906e+01
min,-9.600000e+01,0.0,0.0,1.000000e+00,0.000000e+00
25%,-5.400000e+01,0.0,0.0,1.000000e+01,3.000000e+00
50%,-2.800000e+01,0.0,0.0,1.200000e+01,7.000000e+00
75%,-1.300000e+01,0.0,0.0,2.400000e+01,1.400000e+01
max,-1.000000e+00,4231.0,3595.0,9.200000e+01,8.500000e+01


The source dictionary says ordinary `SK_DPD` measures days past due.
`SK_DPD_DEF` applies a tolerance that ignores debts with low loan amounts.
The tolerance amount is not supplied, so we do not reconstruct it or treat
ordinary/adjusted differences as errors.

`CNT_INSTALMENT` is the reported term and can change. Remaining installments can
also increase. These are schedule observations, not verified counts of payments.

التأخير العادي والتأخير بعد السماح للديون الصغيرة حاجتين مختلفتين؛ هنحسبهم منفصلين.


In [9]:
dictionary_path = DATA_RAW / "HomeCredit_columns_description.csv"
if dictionary_path.is_file():
    dictionary = pd.read_csv(dictionary_path, encoding="cp1252")
    display(dictionary.loc[dictionary["Table"].eq("POS_CASH_balance.csv"), ["Row", "Description"]])
    del dictionary

assert df[["SK_ID_PREV", "MONTHS_BALANCE"]].notna().all().all()
assert df["MONTHS_BALANCE"].lt(0).all(), "Investigate observations at/after application time."
for source in ["SK_DPD", "SK_DPD_DEF", "CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE"]:
    values = df[source].dropna()
    assert values.ge(0).all(), f"Investigate negative {source} values."
    assert values.mod(1).eq(0).all(), f"Investigate noninteger {source} values."
    if source.startswith("CNT_"):
        assert values.lt(2**24).all(), "float32 must represent installment integers exactly."
del values
validation_results = {"Source schema, IDs, numeric domains, and pre-application offsets": "PASS"}


,Row,Description
142,SK_ID_PREV,ID of previous credit in Home Credit related t...
143,SK_ID_CURR,ID of loan in our sample
144,MONTHS_BALANCE,Month of balance relative to application date ...
145,CNT_INSTALMENT,Term of previous credit (can change over time)
146,CNT_INSTALMENT_FUTURE,Installments left to pay on the previous credit
147,NAME_CONTRACT_STATUS,Contract status during the month
148,SK_DPD,DPD (days past due) during the month of previo...
149,SK_DPD_DEF,DPD during the month with tolerance (debts wit...


### Duplicate Contract-Month Policy

We report exact duplicate extras and remove only identical repeated rows from
the analytical view. If different rows share a contract-month key, execution
stops for review. We will not select an arbitrary status or DPD value.
The raw CSV always stays unchanged. With the current EDA data, no rows should be removed.


In [10]:
KEY = ["SK_ID_PREV", "MONTHS_BALANCE"]
exact_duplicate_extras = int(df.duplicated().sum())
duplicate_key_extras = int(df.duplicated(KEY).sum())
print("Exact duplicate extra rows:", exact_duplicate_extras)
print("Repeated contract-month extra rows:", duplicate_key_extras)
if exact_duplicate_extras:
    df = df.drop_duplicates().copy()
conflicting_key_rows = int(df.duplicated(KEY, keep=False).sum())
assert conflicting_key_rows == 0, "Conflicting contract-month rows require a documented resolution."
assert len(df) + exact_duplicate_extras == source_shape[0]
assert df["SK_ID_PREV"].nunique() == n_source_contracts
validation_results["Exact duplicates reported; analytical contract-month keys unique"] = "PASS"


Exact duplicate extra rows: 0
Repeated contract-month extra rows: 0


## 4. Understand the Contract-to-Applicant Mapping

The feature relationship is:

`POS monthly rows → SK_ID_PREV contract summary → previous_application mapping → SK_ID_CURR`.

Each previous contract must map to exactly one applicant. We also compare the
optional POS source applicant ID against this mapping **at contract grain**.
No raw monthly rows are merged into applicant data.


In [11]:
assert previous_mapping.notna().all().all()
mapping_conflicts = int(previous_mapping.groupby("SK_ID_PREV")["SK_ID_CURR"].nunique().gt(1).sum())
duplicate_mapping_ids = int(previous_mapping["SK_ID_PREV"].duplicated().sum())
assert mapping_conflicts == 0, "A previous contract maps to conflicting applicants."
assert duplicate_mapping_ids == 0, "Review repeated IDs in the previous-application mapping."
previous_index = previous_mapping.set_index("SK_ID_PREV")
pos_ids = pd.Index(df["SK_ID_PREV"].unique())
matched_ids = pos_ids.intersection(previous_index.index)
unmatched_ids = pos_ids.difference(previous_index.index)
previous_without_pos = previous_index.index.difference(pos_ids)
n_mapped_pos_applicants = previous_index.loc[matched_ids, "SK_ID_CURR"].nunique()
n_applicants = previous_mapping["SK_ID_CURR"].nunique()


In [12]:
n_source_applicants = None
if has_source_applicant_id:
    assert df["SK_ID_CURR"].notna().all()
    source_id_groups = df.groupby("SK_ID_PREV")["SK_ID_CURR"]
    source_mapping_conflicts = int(source_id_groups.nunique().gt(1).sum())
    source_applicant_map = source_id_groups.first()
    n_source_applicants = df["SK_ID_CURR"].nunique()
    source_mapping_mismatches = int(
        source_applicant_map.loc[matched_ids].ne(previous_index.loc[matched_ids, "SK_ID_CURR"]).sum()
    )
    assert source_mapping_conflicts == 0
    assert source_mapping_mismatches == 0, "POS and previous-application applicant IDs disagree."
    print("Source observed applicants (audit only):", n_source_applicants)
    print("Source applicant mapping conflicts/mismatches:", source_mapping_conflicts, source_mapping_mismatches)
    del source_id_groups, source_applicant_map
    df = df.drop(columns="SK_ID_CURR")


Source observed applicants (audit only): 337252
Source applicant mapping conflicts/mismatches: 0 0


In [13]:
matched_monthly_mask = df["SK_ID_PREV"].isin(matched_ids)
n_matched_monthly_rows = int(matched_monthly_mask.sum())
n_unmatched_monthly_rows = int((~matched_monthly_mask).sum())
linkage_summary = pd.Series({
    "POS contracts matched": len(matched_ids),
    "POS contracts unmatched": len(unmatched_ids),
    "Previous contracts without observed POS history": len(previous_without_pos),
    "Matched analytical monthly rows": n_matched_monthly_rows,
    "Unmatched analytical monthly rows": n_unmatched_monthly_rows,
    "Applicants with mapped POS history": n_mapped_pos_applicants,
    "Applicants in the complete previous mapping": n_applicants
}, name="Count")
display(linkage_summary.to_frame())
assert len(matched_ids) + len(unmatched_ids) == n_source_contracts
assert len(matched_ids) + len(previous_without_pos) == len(previous_mapping)
assert n_matched_monthly_rows + n_unmatched_monthly_rows == len(df)
validation_results["Unambiguous mapping, source-ID audit, and matched/unmatched conservation"] = "PASS"
del matched_monthly_mask


,Count
POS contracts matched,898903
POS contracts unmatched,37422
Previous contracts without observed POS history,771311
Matched analytical monthly rows,9660797
Unmatched analytical monthly rows,340561
Applicants with mapped POS history,334359
Applicants in the complete previous mapping,338857


### What Does “Without POS History” Mean?

An ID-only mapping includes **all previous applications**, including applications
that may not have produced a POS/CASH loan. We cannot identify all POS products
that have no monthly records from IDs alone.

- `pos_total_previous_contracts`: all previous IDs in the mapping.
- `pos_contracts_with_history`: mapped contracts with POS monthly records; this
  is also the total identifiable POS/CASH contract count, without a duplicate feature.
- `pos_previous_contracts_without_pos_history`: the difference, not a confirmed
  count of POS loans that should have reported history.
- `pos_history_coverage_ratio`: observed POS contracts divided by all previous IDs.

العقد اللي ملوش سجل POS ممكن يكون نوع منتج مختلف أو طلب لم يتحول لقرض.

Unmatched POS contracts stay in the contract-level table with missing mapped
applicant IDs and in `unmatched_pos_contracts` for inspection. Even though the
current CSV contains applicant IDs, we do not use them to bypass the required
mapping. These histories cannot contribute to the saved applicant features.


## 5. Group Monthly Records by Previous Contract

We sort months from oldest to newest **inside each contract**. `groupby.diff()`
and `shift()` will then compare earlier and later observations without crossing
contract boundaries. Sorting once avoids repeatedly sorting ten million rows.


In [14]:
df = df.sort_values(KEY, ignore_index=True)
grouped = df.groupby("SK_ID_PREV", sort=True, observed=True)
print("Analytical monthly rows:", len(df))
print("Contract groups:", grouped.ngroups)


Analytical monthly rows: 10001358
Contract groups: 936325


## 6. Engineer Contract-Level Features

### A. History Coverage and Duration

`size` counts monthly records; `nunique` counts distinct observed offsets.
With unique contract-month keys these should agree. The inclusive history span
is `latest - oldest + 1`. Gaps are span minus distinct observed months.

For example, offsets `[-5, -3, -2]` give three observations, a four-month span,
and one missing month. A gap is missing reporting, not proof of delinquency.

عدد الشهور المسجلة مش لازم يساوي طول الفترة، لأن ممكن يكون فيه شهور ناقصة.


In [15]:
contract_features = grouped.agg(
    pos_monthly_records=("MONTHS_BALANCE", "size"),
    pos_observed_months=("MONTHS_BALANCE", "nunique"),
    pos_oldest_month=("MONTHS_BALANCE", "min"),
    pos_latest_month=("MONTHS_BALANCE", "max")
)
contract_features["pos_history_span_months"] = (
    contract_features["pos_latest_month"] - contract_features["pos_oldest_month"] + 1
)
contract_features["pos_gap_months"] = (
    contract_features["pos_history_span_months"] - contract_features["pos_observed_months"]
)
contract_features["pos_has_history_gaps"] = contract_features["pos_gap_months"].gt(0)
display(contract_features.head())
display(contract_features[["pos_observed_months", "pos_history_span_months", "pos_gap_months"]].describe())


,pos_monthly_records,pos_observed_months,pos_oldest_month,pos_latest_month,pos_history_span_months,pos_gap_months,pos_has_history_gaps
SK_ID_PREV,,,,,,,
1000001,3,3,-10,-8,3,0,False
1000002,5,5,-54,-50,5,0,False
1000003,4,4,-4,-1,4,0,False
1000004,8,8,-29,-22,8,0,False
1000005,11,11,-56,-46,11,0,False


,pos_observed_months,pos_history_span_months,pos_gap_months
count,936325.000000,936325.000000,936325.000000
mean,10.681503,10.688926,0.007424
std,6.956280,6.958416,0.241011
min,1.000000,1.000000,0.000000
25%,7.000000,7.000000,0.000000
50%,10.000000,10.000000,0.000000
75%,13.000000,13.000000,0.000000
max,96.000000,96.000000,54.000000


### B. Latest Contract Snapshot and Status

After sorting, `tail(1)` returns the entire final observed row of each group.
We avoid `groupby.last()`, which can skip missing values separately in each
column and accidentally mix different snapshots. Latest installment counts
remain missing when they are missing in that final row.

Latest means **latest observed**, not necessarily the state at application time.
We count contracts in each latest status later, rather than counting monthly statuses.


In [16]:
latest_snapshot = grouped.tail(1).set_index("SK_ID_PREV").sort_index()
latest_state = latest_snapshot[["NAME_CONTRACT_STATUS", "CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE"]].rename(
    columns={
        "NAME_CONTRACT_STATUS": "pos_latest_status",
        "CNT_INSTALMENT": "pos_latest_total_instalments",
        "CNT_INSTALMENT_FUTURE": "pos_latest_remaining_instalments"
    }
)
contract_features = contract_features.join(latest_state, validate="one_to_one")
latest_status_counts = contract_features["pos_latest_status"].value_counts(dropna=False)
display(latest_status_counts.rename("latest_contracts").to_frame())


,latest_contracts
pos_latest_status,
Completed,698421
Active,236149
Signed,1272
Returned to the store,304
Demand,102
Approved,58
Amortized debt,17
Canceled,2
XNA,0


In [17]:
LATEST_STATUS_FLAGS = {
    "Active": "pos_is_latest_active",
    "Completed": "pos_is_latest_completed",
    "Demand": "pos_is_latest_demand",
    "Amortized debt": "pos_is_latest_amortized_debt"
}
for status, flag in LATEST_STATUS_FLAGS.items():
    contract_features[flag] = contract_features["pos_latest_status"].eq(status)
contract_features["pos_is_latest_unknown"] = (
    contract_features["pos_latest_status"].isna() | contract_features["pos_latest_status"].isin(["XNA", "XAP"])
)
contract_features["pos_is_latest_other"] = (
    ~contract_features["pos_latest_status"].isin(LATEST_STATUS_FLAGS)
    & ~contract_features["pos_is_latest_unknown"]
)


### C. Ordinary and Adjusted Delinquency: Frequency and Severity

We use the same operations twice, with separate names: `pos_dpd_...` for `SK_DPD`
and `pos_dpd_def_...` for `SK_DPD_DEF`. A short loop keeps their definitions consistent.

- **Any**: at least one observed positive month.
- **Repeated**: at least two positive months, even if they are separated.
- **Severe**: at least one observed DPD ≥ 90; this threshold is not a default label.
- **Rate**: positive months / months with a known value in that DPD field.
- **Mean/max**: include observed zero days; exclude missing values.

A single positive row is not evidence of persistent delinquency. Missing DPD
does not become a zero-day observation. For a contract with no known DPD, its
event counts, flags, and rates remain missing; known/missing counts describe availability.


In [18]:
DPD_FAMILIES = {"SK_DPD": "pos_dpd", "SK_DPD_DEF": "pos_dpd_def"}
source_dpd_audit = {}
for source, prefix in DPD_FAMILIES.items():
    source_dpd_audit[prefix] = {
        "observed_months": int(df[source].count()),
        "missing_months": int(df[source].isna().sum()),
        "positive_months": int(df[source].gt(0).sum()),
        "days_sum": df[source].sum(min_count=1),
        "severe_contracts": df.loc[df[source].ge(90).fillna(False), "SK_ID_PREV"].nunique()
    }
    contract_features[prefix + "_observed_months"] = grouped[source].count()
    contract_features[prefix + "_missing_months"] = (
        contract_features["pos_monthly_records"] - contract_features[prefix + "_observed_months"]
    )
    contract_features[prefix + "_days_sum"] = grouped[source].sum(min_count=1)
    contract_features[prefix + "_mean"] = grouped[source].mean()
    contract_features[prefix + "_max"] = grouped[source].max()


The temporary Boolean flag identifies positive **observed** rows. `fillna(False)`
applies only to that detection flag; it never fills the raw DPD column. We then
mask a contract's count back to missing if it had no known DPD at all.
`reindex` aligns each grouped result with the contract IDs explicitly.


In [19]:
for source, prefix in DPD_FAMILIES.items():
    df["is_positive_dpd"] = df[source].gt(0).fillna(False).astype("bool")
    positive_counts = grouped["is_positive_dpd"].sum().reindex(contract_features.index)
    no_known_dpd = contract_features[prefix + "_observed_months"].eq(0)
    contract_features[prefix + "_positive_months"] = positive_counts.astype("float32").mask(no_known_dpd)
    contract_features[prefix + "_has_delinquency"] = positive_counts.gt(0).astype("float32").mask(no_known_dpd)
    contract_features[prefix + "_has_repeated"] = positive_counts.ge(2).astype("float32").mask(no_known_dpd)
    contract_features[prefix + "_has_ge90"] = contract_features[prefix + "_max"].ge(90).astype("float32").mask(no_known_dpd)
    contract_features[prefix + "_positive_rate"] = (
        contract_features[prefix + "_positive_months"]
        / contract_features[prefix + "_observed_months"].replace(0, np.nan)
    )
del df["is_positive_dpd"]


### D. Consecutive Delinquent Months

Two positive rows are consecutive only when their offsets differ by **one** and
both DPD values are known. Offsets `[-5, -3]` are repeated observations, but not
consecutive months. We store evaluable-pair coverage as well as the positive-pair flag.
A contract with no evaluable consecutive pair has a missing consecutive flag.
This distinguishes insufficient observation from an observed pair without delinquency.


In [20]:
month_step = grouped["MONTHS_BALANCE"].diff()
source_consecutive_audit = {}
for source, prefix in DPD_FAMILIES.items():
    prior_dpd = grouped[source].shift()
    evaluable_pair = month_step.eq(1) & df[source].notna() & prior_dpd.notna()
    consecutive_positive = evaluable_pair & df[source].gt(0).fillna(False) & prior_dpd.gt(0).fillna(False)
    df["evaluable_pair"] = evaluable_pair
    df["positive_pair"] = consecutive_positive.astype("bool")
    pair_counts = grouped["evaluable_pair"].sum()
    positive_pair_counts = grouped["positive_pair"].sum()
    contract_features[prefix + "_consecutive_evaluable_pairs"] = pair_counts
    contract_features[prefix + "_has_consecutive"] = (
        positive_pair_counts.gt(0).astype("float32").mask(pair_counts.eq(0))
    )
    source_consecutive_audit[prefix] = df.loc[consecutive_positive, "SK_ID_PREV"].nunique()
del df["evaluable_pair"], df["positive_pair"]
del prior_dpd, evaluable_pair, consecutive_positive, pair_counts, positive_pair_counts


### E. Recent Delinquency in a Relative Observation Window

We choose **`-6 ≤ MONTHS_BALANCE ≤ -1`**, the six monthly offsets immediately
before the current application. This is a documented engineering choice, not a
calendar period or six guaranteed observations. We keep its observed-record
coverage so old or incomplete histories are visible.

The most recent positive month is the **maximum signed offset** among positive
observations. It stays missing when no positive month is observed. A recent
rate stays missing if the window contains no known DPD; an observed window
with only zero DPD has a genuine zero rate.

الشهر الأقرب للصفر هو الأحدث. عدم وجود بيانات في آخر ست شهور مش معناه عدم وجود تأخير.


In [21]:
RECENT_MIN_MONTH, RECENT_MAX_MONTH = -6, -1
recent_mask = df["MONTHS_BALANCE"].between(RECENT_MIN_MONTH, RECENT_MAX_MONTH)
df["is_recent_month"] = recent_mask
contract_features["pos_recent_6m_records"] = grouped["is_recent_month"].sum()
source_recent_audit = {}
for source, prefix in DPD_FAMILIES.items():
    df["recent_known_dpd"] = recent_mask & df[source].notna()
    df["recent_positive_dpd"] = recent_mask & df[source].gt(0).fillna(False)
    df["positive_month_offset"] = df["MONTHS_BALANCE"].where(df[source].gt(0).fillna(False))
    recent_known = grouped["recent_known_dpd"].sum()
    recent_positive = grouped["recent_positive_dpd"].sum()
    contract_features[prefix + "_recent_6m_observed_months"] = recent_known
    contract_features[prefix + "_recent_6m_positive_months"] = recent_positive.astype("float32").mask(recent_known.eq(0))
    contract_features[prefix + "_last_positive_month"] = grouped["positive_month_offset"].max()
    source_recent_audit[prefix] = {"observed": int(df["recent_known_dpd"].sum()), "positive": int(df["recent_positive_dpd"].sum())}
del df["is_recent_month"], df["recent_known_dpd"], df["recent_positive_dpd"], df["positive_month_offset"]
del recent_mask, recent_known, recent_positive


### F. Installment Progress and Missing Counts

We summarize observed remaining counts and changes in total terms. Latest counts
already come from the final snapshot. `count()` excludes missing observations;
subtracting it from monthly records gives missingness without conflating missing
counts with a recorded zero.

A changing total term is more than one distinct observed `CNT_INSTALMENT` value.
It can reflect a revised schedule. We do not subtract remaining from total and
call the result verified payments.


In [22]:
installment_features = grouped.agg(
    pos_total_instalments_observed=("CNT_INSTALMENT", "count"),
    pos_remaining_instalments_observed=("CNT_INSTALMENT_FUTURE", "count"),
    pos_total_instalments_distinct=("CNT_INSTALMENT", "nunique"),
    pos_remaining_instalments_mean=("CNT_INSTALMENT_FUTURE", "mean"),
    pos_remaining_instalments_max=("CNT_INSTALMENT_FUTURE", "max")
)
contract_features = contract_features.join(installment_features, validate="one_to_one")
for stem in ["pos_total_instalments", "pos_remaining_instalments"]:
    contract_features[stem + "_missing_months"] = (
        contract_features["pos_monthly_records"] - contract_features[stem + "_observed"]
    )
contract_features["pos_has_changing_total_instalments"] = (
    contract_features["pos_total_instalments_distinct"].gt(1).astype("float32")
    .mask(contract_features["pos_total_instalments_observed"].eq(0))
)
del installment_features


We compare **adjacent observed snapshots** for remaining-count increases. These
comparisons may cross a reporting gap; they do not establish the month of a change.
Both remaining counts must be known. Contracts without a usable pair get a
missing increase flag. We also report how many increases coincide with an
increasing total term, as the EDA found that this explains many changes.


In [23]:
remaining_delta = grouped["CNT_INSTALMENT_FUTURE"].diff()
total_delta = grouped["CNT_INSTALMENT"].diff()
valid_remaining_pair = month_step.notna() & remaining_delta.notna()
remaining_increase = valid_remaining_pair & remaining_delta.gt(0)
df["valid_remaining_pair"] = valid_remaining_pair
df["remaining_increase"] = remaining_increase
remaining_pair_counts = grouped["valid_remaining_pair"].sum()
increase_counts = grouped["remaining_increase"].sum()
contract_features["pos_remaining_evaluable_pairs"] = remaining_pair_counts
contract_features["pos_has_remaining_increase"] = (
    increase_counts.gt(0).astype("float32").mask(remaining_pair_counts.eq(0))
)
increase_context = pd.Series({
    "All observed remaining-count increases": int(remaining_increase.sum()),
    "Total term also increased": int((remaining_increase & total_delta.gt(0)).sum()),
    "Total term unchanged": int((remaining_increase & total_delta.eq(0)).sum()),
    "Total term decreased": int((remaining_increase & total_delta.lt(0)).sum()),
    "Total-term comparison unavailable": int((remaining_increase & total_delta.isna()).sum())
}, name="Pairs")
display(increase_context.to_frame())
assert increase_context.iloc[1:].sum() == increase_context.iloc[0]
del df["valid_remaining_pair"], df["remaining_increase"]
del remaining_delta, total_delta, valid_remaining_pair, remaining_increase, remaining_pair_counts, increase_counts


,Pairs
All observed remaining-count increases,2814
Total term also increased,2402
Total term unchanged,412
Total term decreased,0
Total-term comparison unavailable,0


### G. Data Quality and Availability

Ordinary and adjusted DPD disagreement is evaluated only when both values are
known. We keep their comparison denominator. Adjusted DPD greater than ordinary
DPD is reported for review rather than silently corrected.

`remaining > total` is a review flag. At the latest snapshot we also flag
Completed with positive remaining and Active with zero remaining. These are
status/schedule combinations requiring context, not automatic errors. Zero
remaining installments can coexist with DPD, so it is not a clean-history flag.


In [24]:
paired_dpd = df[["SK_DPD", "SK_DPD_DEF"]].notna().all(axis=1)
df["dpd_comparable"] = paired_dpd
df["dpd_disagreement"] = paired_dpd & df["SK_DPD"].ne(df["SK_DPD_DEF"]).fillna(False)
contract_features["pos_dpd_comparable_months"] = grouped["dpd_comparable"].sum()
contract_features["pos_dpd_disagreement_months"] = (
    grouped["dpd_disagreement"].sum().astype("float32")
    .mask(contract_features["pos_dpd_comparable_months"].eq(0))
)
contract_features["pos_has_dpd_disagreement"] = (
    contract_features["pos_dpd_disagreement_months"].gt(0).astype("float32")
    .mask(contract_features["pos_dpd_comparable_months"].eq(0))
)
dpd_disagreement_rows = int(df["dpd_disagreement"].sum())
adjusted_greater_rows = int((paired_dpd & df["SK_DPD_DEF"].gt(df["SK_DPD"]).fillna(False)).sum())
print("DPD disagreement rows:", dpd_disagreement_rows)
print("Adjusted DPD greater than ordinary DPD (review):", adjusted_greater_rows)
del df["dpd_comparable"], df["dpd_disagreement"]
del paired_dpd


DPD disagreement rows: 183880
Adjusted DPD greater than ordinary DPD (review): 0


In [25]:
both_instalments = df[["CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE"]].notna().all(axis=1)
df["instalment_comparable"] = both_instalments
df["remaining_exceeds_total"] = both_instalments & df["CNT_INSTALMENT_FUTURE"].gt(df["CNT_INSTALMENT"])
contract_features["pos_instalment_comparable_months"] = grouped["instalment_comparable"].sum()
contract_features["pos_has_remaining_exceeds_total"] = (
    grouped["remaining_exceeds_total"].max().astype("float32")
    .mask(contract_features["pos_instalment_comparable_months"].eq(0))
)
latest_remaining = contract_features["pos_latest_remaining_instalments"]
contract_features["pos_latest_zero_remaining"] = latest_remaining.eq(0).astype("float32").mask(latest_remaining.isna())
contract_features["pos_latest_completed_with_remaining"] = contract_features["pos_is_latest_completed"] & latest_remaining.gt(0)
contract_features["pos_latest_active_zero_remaining"] = contract_features["pos_is_latest_active"] & latest_remaining.eq(0)
for source, prefix in DPD_FAMILIES.items():
    contract_features[prefix + "_latest_zero_remaining_positive"] = (
        latest_remaining.eq(0) & latest_snapshot[source].gt(0).fillna(False)
    ).astype("bool")
remaining_exceeds_total_rows = int(df["remaining_exceeds_total"].sum())
print("Monthly rows with remaining > total:", remaining_exceeds_total_rows)
del df["remaining_exceeds_total"]
del df["instalment_comparable"]
del both_instalments, latest_remaining


Monthly rows with remaining > total: 7420


### Validate Contract-Level Aggregation

We check all contract IDs, source-record conservation, latest snapshots, and
ordinary/adjusted DPD totals separately. A second gap calculation sums missing
offsets between adjacent observations, independently of the span formula.

We then inspect a reproducible sample plus boundary cases directly from the
monthly rows. These checks use small arrays and sets, which makes the expected
result easier to understand without an expensive Python loop over every contract.


In [26]:
assert contract_features.index.is_unique
assert len(contract_features) == n_source_contracts
assert contract_features["pos_monthly_records"].sum() == len(df)
assert contract_features["pos_monthly_records"].eq(contract_features["pos_observed_months"]).all()
assert contract_features["pos_gap_months"].ge(0).all()
df["gap_between_observations"] = month_step.sub(1).clip(lower=0).fillna(0)
np.testing.assert_array_equal(contract_features["pos_gap_months"], grouped["gap_between_observations"].sum())
del df["gap_between_observations"]
expected_latest = df.loc[grouped["MONTHS_BALANCE"].idxmax()].set_index("SK_ID_PREV").sort_index()
pd.testing.assert_frame_equal(latest_snapshot, expected_latest, check_exact=True)
np.testing.assert_array_equal(latest_snapshot["MONTHS_BALANCE"], contract_features["pos_latest_month"])
print("Contracts with history gaps:", int(contract_features["pos_has_history_gaps"].sum()))
validation_results["One row per contract, source month totals, latest full snapshots, and independent gaps"] = "PASS"
del expected_latest, month_step


Contracts with history gaps: 1950


In [27]:
for source, prefix in DPD_FAMILIES.items():
    audit = source_dpd_audit[prefix]
    assert contract_features[prefix + "_observed_months"].sum() == audit["observed_months"]
    assert contract_features[prefix + "_missing_months"].sum() == audit["missing_months"]
    assert contract_features[prefix + "_positive_months"].sum(min_count=1) == audit["positive_months"]
    assert contract_features[prefix + "_days_sum"].sum(min_count=1) == audit["days_sum"]
    assert contract_features[prefix + "_has_ge90"].sum() == audit["severe_contracts"]
    assert contract_features[prefix + "_has_consecutive"].sum() == source_consecutive_audit[prefix]
    known = contract_features[prefix + "_observed_months"]
    assert contract_features[prefix + "_positive_months"].dropna().ge(0).all()
    assert (contract_features[prefix + "_positive_months"].isna() | contract_features[prefix + "_positive_months"].le(known)).all()
    assert contract_features[prefix + "_positive_rate"].dropna().between(0, 1).all()
    assert contract_features[prefix + "_positive_rate"].isna().eq(known.eq(0)).all()
    np.testing.assert_allclose(
        contract_features[prefix + "_mean"].to_numpy(dtype="float64", na_value=np.nan),
        (contract_features[prefix + "_days_sum"] / known.replace(0, np.nan)).to_numpy(dtype="float64", na_value=np.nan),
        equal_nan=True
    )
    validation_results[source + " monthly totals, severity, consecutive counts, means, and rate bounds"] = "PASS"


In [28]:
check_contract_ids = contract_features.sample(20, random_state=42).index.tolist()
boundary_columns = [
    "pos_has_history_gaps", "pos_dpd_has_repeated", "pos_dpd_def_has_repeated",
    "pos_dpd_has_consecutive", "pos_dpd_def_has_consecutive",
    "pos_has_changing_total_instalments", "pos_has_remaining_increase",
    "pos_total_instalments_missing_months", "pos_remaining_instalments_missing_months",
    "pos_has_remaining_exceeds_total"
]
for column in boundary_columns:
    check_contract_ids += contract_features.loc[contract_features[column].gt(0), :].head(2).index.tolist()
for column in ["pos_latest_total_instalments", "pos_latest_remaining_instalments"]:
    check_contract_ids += contract_features.loc[contract_features[column].isna()].head(2).index.tolist()
for prefix in DPD_FAMILIES.values():
    check_contract_ids += contract_features.nlargest(2, prefix + "_max").index.tolist()
check_contract_ids = sorted(set(check_contract_ids))
check_contract_rows = df.loc[df["SK_ID_PREV"].isin(check_contract_ids)].copy()


In [29]:
for contract_id, records in check_contract_rows.groupby("SK_ID_PREV"):
    actual = contract_features.loc[contract_id]
    months = records["MONTHS_BALANCE"].to_numpy()
    missing_offsets = set(range(int(months.min()), int(months.max()) + 1)) - set(months)
    assert actual["pos_monthly_records"] == len(records)
    assert actual["pos_gap_months"] == len(missing_offsets)
    for source, prefix in DPD_FAMILIES.items():
        days = records[source].to_numpy(dtype="float64", na_value=np.nan)
        known = np.isfinite(days)
        positive = days > 0
        eligible = (np.diff(months) == 1) & known[:-1] & known[1:]
        consecutive = eligible & positive[:-1] & positive[1:]
        recent = (months >= RECENT_MIN_MONTH) & (months <= RECENT_MAX_MONTH) & known
        expected = {
            "_positive_months": positive.sum() if known.any() else np.nan,
            "_mean": days[known].mean() if known.any() else np.nan,
            "_max": days[known].max() if known.any() else np.nan,
            "_positive_rate": positive.sum() / known.sum() if known.any() else np.nan,
            "_has_repeated": float(positive.sum() >= 2) if known.any() else np.nan,
            "_has_consecutive": float(consecutive.any()) if eligible.any() else np.nan,
            "_recent_6m_observed_months": recent.sum(),
            "_recent_6m_positive_months": (recent & positive).sum() if recent.any() else np.nan,
            "_last_positive_month": months[positive].max() if positive.any() else np.nan
        }
        for suffix, value in expected.items():
            np.testing.assert_allclose(actual[prefix + suffix], value, equal_nan=True)
validation_results["Random and boundary contract checks directly against monthly DPD and offsets"] = "PASS"


In [30]:
for contract_id, records in check_contract_rows.groupby("SK_ID_PREV"):
    actual = contract_features.loc[contract_id]
    totals = records["CNT_INSTALMENT"].to_numpy(dtype="float64")
    remaining = records["CNT_INSTALMENT_FUTURE"].to_numpy(dtype="float64")
    known_total, known_remaining = np.isfinite(totals), np.isfinite(remaining)
    valid_pairs = known_remaining[:-1] & known_remaining[1:]
    expected_increase = float((np.diff(remaining)[valid_pairs] > 0).any()) if valid_pairs.any() else np.nan
    expected_change = float(len(np.unique(totals[known_total])) > 1) if known_total.any() else np.nan
    np.testing.assert_allclose(actual["pos_latest_total_instalments"], totals[-1], equal_nan=True)
    np.testing.assert_allclose(actual["pos_latest_remaining_instalments"], remaining[-1], equal_nan=True)
    np.testing.assert_allclose(actual["pos_has_remaining_increase"], expected_increase, equal_nan=True)
    np.testing.assert_allclose(actual["pos_has_changing_total_instalments"], expected_change, equal_nan=True)
    np.testing.assert_allclose(actual["pos_remaining_instalments_mean"], remaining[known_remaining].mean() if known_remaining.any() else np.nan, rtol=1e-6, equal_nan=True)
    assert actual["pos_total_instalments_missing_months"] == (~known_total).sum()
    assert actual["pos_remaining_instalments_missing_months"] == (~known_remaining).sum()
assert contract_features["pos_has_remaining_increase"].isna().eq(contract_features["pos_remaining_evaluable_pairs"].eq(0)).all()
assert contract_features["pos_has_changing_total_instalments"].isna().eq(contract_features["pos_total_instalments_observed"].eq(0)).all()
assert contract_features["pos_total_instalments_missing_months"].sum() == df["CNT_INSTALMENT"].isna().sum()
assert contract_features["pos_remaining_instalments_missing_months"].sum() == df["CNT_INSTALMENT_FUTURE"].isna().sum()
validation_results["Latest installments, changing terms, evaluable increases, and missingness"] = "PASS"
print("Contracts checked directly:", len(check_contract_ids))


Contracts checked directly: 38


### Keep a Small Source Sample, Then Release Monthly Data

We retain **all monthly rows** for a small set of mapped applicants, including
the boundary-case applicants above. This will allow independent applicant checks.
After the population and contract checks, releasing the full monthly table
reduces memory while building applicant features.


In [31]:
check_applicant_ids = previous_index.reindex(check_contract_ids)["SK_ID_CURR"].dropna().astype("int32").tolist()
check_applicant_ids += previous_mapping["SK_ID_CURR"].drop_duplicates().sample(10, random_state=42).tolist()
check_applicant_ids = sorted(set(check_applicant_ids))
check_previous_mapping = previous_mapping.loc[previous_mapping["SK_ID_CURR"].isin(check_applicant_ids)].copy()
check_applicant_rows = df.loc[df["SK_ID_PREV"].isin(check_previous_mapping["SK_ID_PREV"])].copy()
analytical_shape = df.shape
del grouped, df, latest_snapshot, latest_state, check_contract_rows
gc.collect()


0

## 7. Map Contract Summaries and Aggregate by Applicant

`map()` looks up the applicant ID for each **contract summary**, not each monthly
row. Unmatched contracts get a missing mapped ID. We preserve them separately
and verify their monthly counts before selecting the mapped subset.


In [32]:
for column in contract_features.select_dtypes(include="integer").columns:
    contract_features[column] = contract_features[column].astype("Int32" if contract_features[column].isna().any() else "int32")
contract_features["SK_ID_CURR"] = contract_features.index.map(previous_index["SK_ID_CURR"]).astype("Int32")
contract_features = contract_features.reset_index()
unmatched_pos_contracts = contract_features.loc[contract_features["SK_ID_CURR"].isna()].copy()
mapped_contracts = contract_features.loc[contract_features["SK_ID_CURR"].notna()].copy()
mapped_contracts["SK_ID_CURR"] = mapped_contracts["SK_ID_CURR"].astype("int32")
assert contract_features["SK_ID_PREV"].is_unique
assert len(unmatched_pos_contracts) == len(unmatched_ids)
assert len(mapped_contracts) == len(matched_ids)
assert unmatched_pos_contracts["pos_monthly_records"].sum() == n_unmatched_monthly_rows
assert mapped_contracts["pos_monthly_records"].sum() == n_matched_monthly_rows
assert mapped_contracts["SK_ID_CURR"].nunique() == n_mapped_pos_applicants
display(unmatched_pos_contracts[["SK_ID_PREV", "SK_ID_CURR", "pos_monthly_records", "pos_latest_status"]].head())
pos_grouped = mapped_contracts.groupby("SK_ID_CURR", sort=True)


,SK_ID_PREV,SK_ID_CURR,pos_monthly_records,pos_latest_status
18,1000021,<NA>,5,Completed
31,1000037,<NA>,6,Completed
35,1000041,<NA>,12,Completed
84,1000095,<NA>,7,Completed
91,1000104,<NA>,7,Completed


### A. Contract Counts, Latest Status, and Coverage

We start with all applicants in the previous mapping, as the Bureau notebook
starts with its account table. The status summary contains only observed POS
contracts. A left merge keeps applicants without observed POS history.

Active and Completed ratios use **observed POS contracts** as their denominator.
They do not use the number of monthly rows or all previous applications.
Demand and Amortized debt are retained separately because the EDA found distinctive
DPD patterns; other known statuses share one count. Unknown status is separate.


In [33]:
pos_count_features = previous_mapping.groupby("SK_ID_CURR").agg(
    pos_total_previous_contracts=("SK_ID_PREV", "size")
).reset_index()
observed_status_features = pos_grouped.agg(
    pos_contracts_with_history=("SK_ID_PREV", "size"),
    pos_latest_active_contracts=("pos_is_latest_active", "sum"),
    pos_latest_completed_contracts=("pos_is_latest_completed", "sum"),
    pos_latest_demand_contracts=("pos_is_latest_demand", "sum"),
    pos_latest_amortized_debt_contracts=("pos_is_latest_amortized_debt", "sum"),
    pos_latest_other_status_contracts=("pos_is_latest_other", "sum"),
    pos_latest_unknown_status_contracts=("pos_is_latest_unknown", "sum")
).reset_index()
STATUS_COUNT_COLUMNS = [column for column in observed_status_features if column != "SK_ID_CURR"]
pos_count_features = pos_count_features.merge(observed_status_features, on="SK_ID_CURR", how="left", validate="one_to_one")
pos_count_features[STATUS_COUNT_COLUMNS] = pos_count_features[STATUS_COUNT_COLUMNS].fillna(0).astype("int32")
pos_count_features["pos_previous_contracts_without_pos_history"] = (
    pos_count_features["pos_total_previous_contracts"] - pos_count_features["pos_contracts_with_history"]
)


In [34]:
pos_count_features["pos_history_coverage_ratio"] = pos_count_features["pos_contracts_with_history"] / pos_count_features["pos_total_previous_contracts"]
observed_contract_denominator = pos_count_features["pos_contracts_with_history"].replace(0, np.nan)
pos_count_features["pos_active_contract_ratio"] = pos_count_features["pos_latest_active_contracts"] / observed_contract_denominator
pos_count_features["pos_completed_contract_ratio"] = pos_count_features["pos_latest_completed_contracts"] / observed_contract_denominator
RATE_COLUMNS = ["pos_history_coverage_ratio", "pos_active_contract_ratio", "pos_completed_contract_ratio"]
COMPLETE_COUNT_COLUMNS = ["pos_total_previous_contracts", "pos_previous_contracts_without_pos_history"] + STATUS_COUNT_COLUMNS
pos_count_features.head()


,SK_ID_CURR,pos_total_previous_contracts,pos_contracts_with_history,pos_latest_active_contracts,pos_latest_completed_contracts,pos_latest_demand_contracts,pos_latest_amortized_debt_contracts,pos_latest_other_status_contracts,pos_latest_unknown_status_contracts,pos_previous_contracts_without_pos_history,pos_history_coverage_ratio,pos_active_contract_ratio,pos_completed_contract_ratio
0,100001,1,1,0,1,0,0,0,0,0,1.0,0.000000,1.000000
1,100002,1,1,1,0,0,0,0,0,0,1.0,1.000000,0.000000
2,100003,3,3,1,2,0,0,0,0,0,1.0,0.333333,0.666667
3,100004,1,1,0,1,0,0,0,0,0,1.0,0.000000,1.000000
4,100005,2,1,0,1,0,0,0,0,1,0.5,0.000000,1.000000


### B. Applicant History Length and Gaps

History-length means give **each observed contract equal weight**. The total
record count describes the amount of source evidence. Span mean/max are separate
from observed-month mean/min/max, so reporting gaps are visible. Latest/oldest
offsets describe the available history, including stale observations.


In [35]:
pos_history_features = pos_grouped.agg(
    pos_monthly_records_total=("pos_monthly_records", "sum"),
    pos_history_months_mean=("pos_observed_months", "mean"),
    pos_history_months_min=("pos_observed_months", "min"),
    pos_history_months_max=("pos_observed_months", "max"),
    pos_history_span_months_mean=("pos_history_span_months", "mean"),
    pos_history_span_months_max=("pos_history_span_months", "max"),
    pos_contracts_with_history_gaps=("pos_has_history_gaps", "sum"),
    pos_gap_months_total=("pos_gap_months", "sum"),
    pos_oldest_observed_month=("pos_oldest_month", "min"),
    pos_latest_observed_month=("pos_latest_month", "max")
).reset_index()
COMPLETE_COUNT_COLUMNS += ["pos_monthly_records_total", "pos_contracts_with_history_gaps", "pos_gap_months_total"]
pos_history_features.head()


,SK_ID_CURR,pos_monthly_records_total,pos_history_months_mean,pos_history_months_min,pos_history_months_max,pos_history_span_months_mean,pos_history_span_months_max,pos_contracts_with_history_gaps,pos_gap_months_total,pos_oldest_observed_month,pos_latest_observed_month
0,100001,5,5.000000,5,5,5.000000,5,0,0,-57,-53
1,100002,19,19.000000,19,19,19.000000,19,0,0,-19,-1
2,100003,28,9.333333,8,12,9.333333,12,0,0,-77,-18
3,100004,4,4.000000,4,4,4.000000,4,0,0,-27,-24
4,100005,11,11.000000,11,11,11.000000,11,0,0,-25,-15


### C. Applicant Delinquency: Two Separate Families

Contract flags are summed to count affected contracts. The applicant DPD mean
is **monthly-observation weighted**: contract day totals / known monthly DPD
observations. The positive-month rate uses the same known-month denominator.
The contract delinquency ratio uses contracts with at least one known DPD.

Longer histories give more opportunities to detect delinquency. History features
and known-month/contract counts provide that observation context. We do not
interpret a larger count alone as higher intrinsic borrower risk.

| Suffix | English explanation | المعنى بالعربي |
|---|---|---|
| `_delinquent_contracts` | Contracts with any observed positive DPD | عقود فيها شهر تأخير واحد على الأقل |
| `_repeated_contracts` | Contracts with at least two positive months | عقود فيها تأخير في شهرين أو أكثر |
| `_consecutive_contracts` | Contracts with a positive adjacent-month pair | عقود فيها شهرين متتاليين بتأخير |
| `_ge90_contracts` | Contracts reaching 90 observed DPD | عقود وصل تأخيرها لـ 90 يوم |
| `_positive_rate` | Positive known months / all known months | نسبة الشهور المتأخرة من الشهور المعروفة |

Insufficient consecutive-pair coverage remains distinct from zero detected
consecutive delinquency. Both DPD families have their own availability features.


In [36]:
dpd_feature_groups = {}
for source, prefix in DPD_FAMILIES.items():
    mapped_contracts[prefix + "_has_observed"] = mapped_contracts[prefix + "_observed_months"].gt(0)
    mapped_contracts[prefix + "_has_missing"] = mapped_contracts[prefix + "_missing_months"].gt(0)
    mapped_contracts[prefix + "_has_evaluable_consecutive"] = mapped_contracts[prefix + "_consecutive_evaluable_pairs"].gt(0)
    features = pos_grouped.agg(**{
        prefix + "_observed_months": (prefix + "_observed_months", "sum"),
        prefix + "_missing_months": (prefix + "_missing_months", "sum"),
        prefix + "_observed_contracts": (prefix + "_has_observed", "sum"),
        prefix + "_contracts_with_missing": (prefix + "_has_missing", "sum"),
        prefix + "_delinquent_contracts": (prefix + "_has_delinquency", "sum"),
        prefix + "_repeated_contracts": (prefix + "_has_repeated", "sum"),
        prefix + "_consecutive_contracts": (prefix + "_has_consecutive", "sum"),
        prefix + "_consecutive_evaluable_contracts": (prefix + "_has_evaluable_consecutive", "sum"),
        prefix + "_ge90_contracts": (prefix + "_has_ge90", "sum"),
        prefix + "_positive_months": (prefix + "_positive_months", "sum"),
        prefix + "_days_sum": (prefix + "_days_sum", "sum"),
        prefix + "_max": (prefix + "_max", "max")
    }).reset_index()
    features[prefix + "_mean"] = features[prefix + "_days_sum"] / features[prefix + "_observed_months"].replace(0, np.nan)
    features = features.drop(columns=prefix + "_days_sum")
    dpd_feature_groups[prefix] = features


Pandas `sum()` returns zero for an all-missing group. For DPD measurements we
explicitly restore missingness when no known observations exist. We only keep
zero counts for availability, where zero means an empty observed set.


In [37]:
for prefix, features in dpd_feature_groups.items():
    known_months = features[prefix + "_observed_months"].replace(0, np.nan)
    known_contracts = features[prefix + "_observed_contracts"].replace(0, np.nan)
    for suffix in ["_delinquent_contracts", "_repeated_contracts", "_ge90_contracts", "_positive_months"]:
        features[prefix + suffix] = features[prefix + suffix].mask(known_months.isna())
    features[prefix + "_consecutive_contracts"] = features[prefix + "_consecutive_contracts"].mask(features[prefix + "_consecutive_evaluable_contracts"].eq(0))
    features[prefix + "_positive_rate"] = features[prefix + "_positive_months"] / known_months
    features[prefix + "_contract_delinquency_ratio"] = features[prefix + "_delinquent_contracts"] / known_contracts
    RATE_COLUMNS += [prefix + "_positive_rate", prefix + "_contract_delinquency_ratio"]
    COMPLETE_COUNT_COLUMNS += [prefix + suffix for suffix in ["_observed_months", "_missing_months", "_observed_contracts", "_contracts_with_missing", "_consecutive_evaluable_contracts"]]
display(dpd_feature_groups["pos_dpd"].head())
display(dpd_feature_groups["pos_dpd_def"].head())


,SK_ID_CURR,pos_dpd_observed_months,pos_dpd_missing_months,pos_dpd_observed_contracts,pos_dpd_contracts_with_missing,pos_dpd_delinquent_contracts,pos_dpd_repeated_contracts,pos_dpd_consecutive_contracts,pos_dpd_consecutive_evaluable_contracts,pos_dpd_ge90_contracts,pos_dpd_positive_months,pos_dpd_max,pos_dpd_mean,pos_dpd_positive_rate,pos_dpd_contract_delinquency_ratio
0,100001,5,0,1,0,0.0,0.0,0.0,1,0.0,0.0,0,0.0,0.0,0.0
1,100002,19,0,1,0,0.0,0.0,0.0,1,0.0,0.0,0,0.0,0.0,0.0
2,100003,28,0,3,0,0.0,0.0,0.0,3,0.0,0.0,0,0.0,0.0,0.0
3,100004,4,0,1,0,0.0,0.0,0.0,1,0.0,0.0,0,0.0,0.0,0.0
4,100005,11,0,1,0,0.0,0.0,0.0,1,0.0,0.0,0,0.0,0.0,0.0


,SK_ID_CURR,pos_dpd_def_observed_months,pos_dpd_def_missing_months,pos_dpd_def_observed_contracts,pos_dpd_def_contracts_with_missing,pos_dpd_def_delinquent_contracts,pos_dpd_def_repeated_contracts,pos_dpd_def_consecutive_contracts,pos_dpd_def_consecutive_evaluable_contracts,pos_dpd_def_ge90_contracts,pos_dpd_def_positive_months,pos_dpd_def_max,pos_dpd_def_mean,pos_dpd_def_positive_rate,pos_dpd_def_contract_delinquency_ratio
0,100001,5,0,1,0,0.0,0.0,0.0,1,0.0,0.0,0,0.0,0.0,0.0
1,100002,19,0,1,0,0.0,0.0,0.0,1,0.0,0.0,0,0.0,0.0,0.0
2,100003,28,0,3,0,0.0,0.0,0.0,3,0.0,0.0,0,0.0,0.0,0.0
3,100004,4,0,1,0,0.0,0.0,0.0,1,0.0,0.0,0,0.0,0.0,0.0
4,100005,11,0,1,0,0.0,0.0,0.0,1,0.0,0.0,0,0.0,0.0,0.0


### D. Applicant Recent Delinquency

We sum contract-window observations and positive months. The recent rate is
monthly-record weighted across the applicant's contracts. The same relative
month can contribute once per contract, so this is not a count of distinct
applicant calendar months. The latest positive offset is the maximum across
contracts. The recent indicator is missing when no known window observations exist.


In [38]:
mapped_contracts["pos_has_recent_history"] = mapped_contracts["pos_recent_6m_records"].gt(0)
pos_recent_features = pos_grouped.agg(
    pos_recent_6m_monthly_records=("pos_recent_6m_records", "sum"),
    pos_recent_6m_observed_contracts=("pos_has_recent_history", "sum")
).reset_index()
for source, prefix in DPD_FAMILIES.items():
    recent = pos_grouped.agg(**{
        prefix + "_recent_6m_observed_months": (prefix + "_recent_6m_observed_months", "sum"),
        prefix + "_recent_6m_positive_months": (prefix + "_recent_6m_positive_months", "sum"),
        prefix + "_last_positive_month": (prefix + "_last_positive_month", "max")
    }).reset_index()
    denominator = recent[prefix + "_recent_6m_observed_months"].replace(0, np.nan)
    recent[prefix + "_recent_6m_positive_months"] = recent[prefix + "_recent_6m_positive_months"].mask(denominator.isna())
    recent[prefix + "_recent_6m_positive_rate"] = recent[prefix + "_recent_6m_positive_months"] / denominator
    recent[prefix + "_has_recent_6m_delinquency"] = recent[prefix + "_recent_6m_positive_months"].gt(0).astype("float32").mask(denominator.isna())
    pos_recent_features = pos_recent_features.merge(recent, on="SK_ID_CURR", how="left", validate="one_to_one")
    RATE_COLUMNS.append(prefix + "_recent_6m_positive_rate")
    COMPLETE_COUNT_COLUMNS.append(prefix + "_recent_6m_observed_months")
COMPLETE_COUNT_COLUMNS += ["pos_recent_6m_monthly_records", "pos_recent_6m_observed_contracts"]
pos_recent_features.head()


,SK_ID_CURR,pos_recent_6m_monthly_records,pos_recent_6m_observed_contracts,pos_dpd_recent_6m_observed_months,pos_dpd_recent_6m_positive_months,pos_dpd_last_positive_month,pos_dpd_recent_6m_positive_rate,pos_dpd_has_recent_6m_delinquency,pos_dpd_def_recent_6m_observed_months,pos_dpd_def_recent_6m_positive_months,pos_dpd_def_last_positive_month,pos_dpd_def_recent_6m_positive_rate,pos_dpd_def_has_recent_6m_delinquency
0,100001,0,0,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
1,100002,6,1,6,0.0,NaN,0.0,0.0,6,0.0,NaN,0.0,0.0
2,100003,0,0,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
3,100004,0,0,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
4,100005,0,0,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN


### E. Applicant Installment Progress

Latest total/remaining means and maxima use each contract's **latest snapshot**.
`pos_remaining_instalments_contract_mean` averages each contract's historical
remaining-count mean, giving equal weight to contracts with usable values.
Its maximum companion is the largest remaining count seen anywhere in the history.

Missing total/remaining contract counts and latest missing counts describe
availability. Zero remaining is counted only when the latest count is known.
Increase counts are accompanied by the number of contracts with an evaluable pair.


In [39]:
mapped_contracts["pos_has_total_instalments"] = mapped_contracts["pos_total_instalments_observed"].gt(0)
mapped_contracts["pos_has_remaining_instalments"] = mapped_contracts["pos_remaining_instalments_observed"].gt(0)
mapped_contracts["pos_has_missing_total_instalments"] = mapped_contracts["pos_total_instalments_missing_months"].gt(0)
mapped_contracts["pos_has_missing_remaining_instalments"] = mapped_contracts["pos_remaining_instalments_missing_months"].gt(0)
mapped_contracts["pos_latest_missing_total_instalments"] = mapped_contracts["pos_latest_total_instalments"].isna()
mapped_contracts["pos_latest_missing_remaining_instalments"] = mapped_contracts["pos_latest_remaining_instalments"].isna()
mapped_contracts["pos_latest_remaining_observed"] = mapped_contracts["pos_latest_remaining_instalments"].notna()
mapped_contracts["pos_remaining_increase_evaluable"] = mapped_contracts["pos_remaining_evaluable_pairs"].gt(0)


In [40]:
pos_installment_features = pos_grouped.agg(
    pos_latest_total_instalments_mean=("pos_latest_total_instalments", "mean"),
    pos_latest_total_instalments_max=("pos_latest_total_instalments", "max"),
    pos_latest_remaining_instalments_mean=("pos_latest_remaining_instalments", "mean"),
    pos_latest_remaining_instalments_max=("pos_latest_remaining_instalments", "max"),
    pos_remaining_instalments_contract_mean=("pos_remaining_instalments_mean", "mean"),
    pos_remaining_instalments_max=("pos_remaining_instalments_max", "max"),
    pos_latest_zero_remaining_contracts=("pos_latest_zero_remaining", "sum"),
    pos_total_instalments_observed_contracts=("pos_has_total_instalments", "sum"),
    pos_remaining_instalments_observed_contracts=("pos_has_remaining_instalments", "sum"),
    pos_contracts_with_missing_total_instalments=("pos_has_missing_total_instalments", "sum"),
    pos_contracts_with_missing_remaining_instalments=("pos_has_missing_remaining_instalments", "sum"),
    pos_latest_missing_total_instalment_contracts=("pos_latest_missing_total_instalments", "sum"),
    pos_latest_missing_remaining_instalment_contracts=("pos_latest_missing_remaining_instalments", "sum"),
    pos_latest_remaining_observed_contracts=("pos_latest_remaining_observed", "sum"),
    pos_changing_total_instalment_contracts=("pos_has_changing_total_instalments", "sum"),
    pos_remaining_increase_contracts=("pos_has_remaining_increase", "sum"),
    pos_remaining_increase_evaluable_contracts=("pos_remaining_increase_evaluable", "sum")
).reset_index()


In [41]:
for feature, availability in {
    "pos_latest_zero_remaining_contracts": "pos_latest_remaining_observed_contracts",
    "pos_changing_total_instalment_contracts": "pos_total_instalments_observed_contracts",
    "pos_remaining_increase_contracts": "pos_remaining_increase_evaluable_contracts"
}.items():
    pos_installment_features[feature] = pos_installment_features[feature].mask(pos_installment_features[availability].eq(0))
COMPLETE_COUNT_COLUMNS += [
    "pos_total_instalments_observed_contracts", "pos_remaining_instalments_observed_contracts",
    "pos_contracts_with_missing_total_instalments", "pos_contracts_with_missing_remaining_instalments",
    "pos_latest_missing_total_instalment_contracts", "pos_latest_missing_remaining_instalment_contracts",
    "pos_latest_remaining_observed_contracts", "pos_remaining_increase_evaluable_contracts"
]
pos_installment_features.head()


,SK_ID_CURR,pos_latest_total_instalments_mean,pos_latest_total_instalments_max,pos_latest_remaining_instalments_mean,pos_latest_remaining_instalments_max,pos_remaining_instalments_contract_mean,pos_remaining_instalments_max,pos_latest_zero_remaining_contracts,pos_total_instalments_observed_contracts,pos_remaining_instalments_observed_contracts,pos_contracts_with_missing_total_instalments,pos_contracts_with_missing_remaining_instalments,pos_latest_missing_total_instalment_contracts,pos_latest_missing_remaining_instalment_contracts,pos_latest_remaining_observed_contracts,pos_changing_total_instalment_contracts,pos_remaining_increase_contracts,pos_remaining_increase_evaluable_contracts
0,100001,4.000000,4.0,0.000000,0.0,2.000000,4.0,1.0,1,1,0,0,0,0,1,0.0,0.0,1
1,100002,24.000000,24.0,6.000000,6.0,15.000000,24.0,0.0,1,1,0,0,0,0,1,0.0,0.0,1
2,100003,8.333333,12.0,0.333333,1.0,5.666667,12.0,2.0,3,3,0,0,0,0,3,1.0,0.0,3
3,100004,3.000000,3.0,0.000000,0.0,2.250000,4.0,1.0,1,1,0,0,0,0,1,1.0,0.0,1
4,100005,9.000000,9.0,0.000000,0.0,7.200000,12.0,1.0,1,1,1,1,0,0,1,1.0,0.0,1


### F. Applicant Quality and DPD Differences

Disagreement frequency uses months with both DPD values known. The status/count
flags below count observed combinations; missing installment snapshots are
reported separately. We retain DPD beside zero remaining counts in **both** DPD
families, because the EDA found that the ordinary and adjusted patterns differ.


In [42]:
pos_quality_features = pos_grouped.agg(
    pos_dpd_comparable_months=("pos_dpd_comparable_months", "sum"),
    pos_dpd_disagreement_months=("pos_dpd_disagreement_months", "sum"),
    pos_dpd_disagreement_contracts=("pos_has_dpd_disagreement", "sum"),
    pos_remaining_exceeds_total_contracts=("pos_has_remaining_exceeds_total", "sum"),
    pos_latest_completed_with_remaining_contracts=("pos_latest_completed_with_remaining", "sum"),
    pos_latest_active_zero_remaining_contracts=("pos_latest_active_zero_remaining", "sum"),
    pos_dpd_latest_zero_remaining_positive_contracts=("pos_dpd_latest_zero_remaining_positive", "sum"),
    pos_dpd_def_latest_zero_remaining_positive_contracts=("pos_dpd_def_latest_zero_remaining_positive", "sum")
).reset_index()
denominator = pos_quality_features["pos_dpd_comparable_months"].replace(0, np.nan)
for feature in ["pos_dpd_disagreement_months", "pos_dpd_disagreement_contracts"]:
    pos_quality_features[feature] = pos_quality_features[feature].mask(denominator.isna())
pos_quality_features["pos_dpd_disagreement_rate"] = pos_quality_features["pos_dpd_disagreement_months"] / denominator
comparable_installment_months = pos_grouped["pos_instalment_comparable_months"].sum()
pos_quality_features["pos_remaining_exceeds_total_contracts"] = pos_quality_features["pos_remaining_exceeds_total_contracts"].mask(comparable_installment_months.to_numpy() == 0)
RATE_COLUMNS.append("pos_dpd_disagreement_rate")
COMPLETE_COUNT_COLUMNS.append("pos_dpd_comparable_months")
pos_quality_features.head()


,SK_ID_CURR,pos_dpd_comparable_months,pos_dpd_disagreement_months,pos_dpd_disagreement_contracts,pos_remaining_exceeds_total_contracts,pos_latest_completed_with_remaining_contracts,pos_latest_active_zero_remaining_contracts,pos_dpd_latest_zero_remaining_positive_contracts,pos_dpd_def_latest_zero_remaining_positive_contracts,pos_dpd_disagreement_rate
0,100001,5,0.0,0.0,0.0,0,0,0,0,0.0
1,100002,19,0.0,0.0,0.0,0,0,0,0,0.0
2,100003,28,0.0,0.0,0.0,0,0,0,0,0.0
3,100004,4,0.0,0.0,0.0,0,0,0,0,0.0
4,100005,11,0.0,0.0,0.0,0,0,0,0,0.0


## 8. Combine Applicant-Level Feature Groups

All feature tables now contain at most one row per applicant. `validate="one_to_one"`
makes Pandas reject an accidental duplicated applicant ID during a merge.
The count table supplies the complete previous-mapping applicant population.

Only the explicitly listed **availability and detected-status counts** receive
zero for applicants with no POS history. Measurements, event frequencies, and
rates remain missing when their required observations are absent.


In [43]:
feature_groups = {
    "Contract counts, latest status, and coverage": pos_count_features,
    "History length and gaps": pos_history_features,
    "Ordinary DPD frequency and severity": dpd_feature_groups["pos_dpd"],
    "Adjusted DPD frequency and severity": dpd_feature_groups["pos_dpd_def"],
    "Recent relative-window delinquency": pos_recent_features,
    "Installment progress and availability": pos_installment_features,
    "Quality and DPD differences": pos_quality_features
}
for name, feature_group in feature_groups.items():
    assert feature_group["SK_ID_CURR"].is_unique, name
    assert feature_group["SK_ID_CURR"].notna().all(), name
    assert len(feature_group) == (n_applicants if feature_group is pos_count_features else n_mapped_pos_applicants), name

pos_cash_features = pos_count_features.copy()
for feature_group in list(feature_groups.values())[1:]:
    pos_cash_features = pos_cash_features.merge(feature_group, on="SK_ID_CURR", how="left", validate="one_to_one")
pos_cash_features = pos_cash_features.sort_values("SK_ID_CURR").reset_index(drop=True)
no_pos_history = pos_cash_features["pos_contracts_with_history"].eq(0)
empty_history_count_columns = [column for column in COMPLETE_COUNT_COLUMNS if column not in ["pos_total_previous_contracts", "pos_previous_contracts_without_pos_history"]]
pos_cash_features.loc[no_pos_history, empty_history_count_columns] = 0


In [44]:
for column in COMPLETE_COUNT_COLUMNS:
    pos_cash_features[column] = pos_cash_features[column].astype("int32")
print("Shape:", pos_cash_features.shape)
print("Unique applicants:", pos_cash_features["SK_ID_CURR"].nunique())
print("Applicants without mapped POS history:", int(no_pos_history.sum()))
pos_cash_features.head()


Shape: (338857, 89)
Unique applicants: 338857
Applicants without mapped POS history: 4498


,SK_ID_CURR,pos_total_previous_contracts,pos_contracts_with_history,pos_latest_active_contracts,pos_latest_completed_contracts,pos_latest_demand_contracts,pos_latest_amortized_debt_contracts,pos_latest_other_status_contracts,pos_latest_unknown_status_contracts,pos_previous_contracts_without_pos_history,...,pos_remaining_increase_evaluable_contracts,pos_dpd_comparable_months,pos_dpd_disagreement_months,pos_dpd_disagreement_contracts,pos_remaining_exceeds_total_contracts,pos_latest_completed_with_remaining_contracts,pos_latest_active_zero_remaining_contracts,pos_dpd_latest_zero_remaining_positive_contracts,pos_dpd_def_latest_zero_remaining_positive_contracts,pos_dpd_disagreement_rate
0,100001,1,1,0,1,0,0,0,0,0,...,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,100002,1,1,1,0,0,0,0,0,0,...,1,19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,100003,3,3,1,2,0,0,0,0,0,...,3,28,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,100004,1,1,0,1,0,0,0,0,0,...,1,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,100005,2,1,0,1,0,0,0,0,1,...,1,11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 9. Validation

We validate applicant IDs, population-wide conservation, independent latest-status
counts, direct source examples, rate formulas, and missingness. This is a feature
table only; we do not merge it into an application table or use target information.


In [45]:
assert pos_cash_features["SK_ID_CURR"].notna().all()
assert pos_cash_features["SK_ID_CURR"].is_unique
assert pos_cash_features["SK_ID_CURR"].duplicated().sum() == 0
assert len(pos_cash_features) == n_applicants
np.testing.assert_array_equal(pos_cash_features["SK_ID_CURR"], np.sort(previous_mapping["SK_ID_CURR"].unique()))
assert pos_cash_features.columns.is_unique
assert all(column.startswith("pos_") for column in pos_cash_features if column != "SK_ID_CURR")
assert "TARGET" not in pos_cash_features.columns
assert pos_cash_features["pos_total_previous_contracts"].sum() == len(previous_mapping)
assert pos_cash_features["pos_contracts_with_history"].sum() == len(matched_ids)
assert pos_cash_features["pos_previous_contracts_without_pos_history"].sum() == len(previous_without_pos)
assert pos_cash_features["pos_monthly_records_total"].sum() == n_matched_monthly_rows
assert pos_cash_features["pos_contracts_with_history"].gt(0).sum() == n_mapped_pos_applicants
assert (pos_cash_features["pos_contracts_with_history"] + pos_cash_features["pos_previous_contracts_without_pos_history"]).eq(pos_cash_features["pos_total_previous_contracts"]).all()
assert pos_cash_features[STATUS_COUNT_COLUMNS[1:]].sum(axis=1).eq(pos_cash_features["pos_contracts_with_history"]).all()
validation_results["Unique applicant grain, exact mapping population, and contract/month conservation"] = "PASS"


An independent cross-tabulation checks latest statuses for every mapped applicant.
Unknown/missing and other statuses are given the same explicitly documented
buckets. The table counts **contract snapshots**, so longer histories cannot
inflate these status counts.


In [46]:
status_bucket = mapped_contracts["pos_latest_status"].astype("string").fillna("Unknown")
status_bucket = status_bucket.mask(status_bucket.isin(["XNA", "XAP"]), "Unknown")
status_bucket = status_bucket.where(status_bucket.isin(["Active", "Completed", "Demand", "Amortized debt", "Unknown"]), "Other")
status_columns = {
    "Active": "pos_latest_active_contracts", "Completed": "pos_latest_completed_contracts",
    "Demand": "pos_latest_demand_contracts", "Amortized debt": "pos_latest_amortized_debt_contracts",
    "Other": "pos_latest_other_status_contracts", "Unknown": "pos_latest_unknown_status_contracts"
}
expected_status = pd.crosstab(mapped_contracts["SK_ID_CURR"], status_bucket)
expected_status = expected_status.reindex(index=pos_cash_features["SK_ID_CURR"], columns=list(status_columns), fill_value=0).rename(columns=status_columns)
pd.testing.assert_frame_equal(
    pos_cash_features.set_index("SK_ID_CURR")[list(status_columns.values())],
    expected_status, check_dtype=False, check_names=False
)
validation_results["Every applicant latest-status count verified independently"] = "PASS"
del expected_status, status_bucket


In [47]:
for source, prefix in DPD_FAMILIES.items():
    for suffix in ["_observed_months", "_missing_months", "_positive_months"]:
        assert pos_cash_features[prefix + suffix].sum(min_count=1) == mapped_contracts[prefix + suffix].sum(min_count=1)
    for feature_suffix, contract_suffix in {
        "_delinquent_contracts": "_has_delinquency", "_repeated_contracts": "_has_repeated",
        "_consecutive_contracts": "_has_consecutive", "_ge90_contracts": "_has_ge90"
    }.items():
        assert pos_cash_features[prefix + feature_suffix].sum(min_count=1) == mapped_contracts[prefix + contract_suffix].sum(min_count=1)
        assert pos_cash_features[prefix + feature_suffix].dropna().ge(0).all()
        assert (pos_cash_features[prefix + feature_suffix].isna() | pos_cash_features[prefix + feature_suffix].le(pos_cash_features[prefix + "_observed_contracts"])).all()
    for suffix in ["_recent_6m_observed_months", "_recent_6m_positive_months"]:
        assert pos_cash_features[prefix + suffix].sum(min_count=1) == mapped_contracts[prefix + suffix].sum(min_count=1)
    assert (pos_cash_features[prefix + "_positive_months"].isna() | pos_cash_features[prefix + "_positive_months"].le(pos_cash_features[prefix + "_observed_months"])).all()
    assert (pos_cash_features[prefix + "_recent_6m_positive_months"].isna() | pos_cash_features[prefix + "_recent_6m_positive_months"].le(pos_cash_features[prefix + "_recent_6m_observed_months"])).all()
    validation_results[source + " applicant counts conserved from mapped contracts"] = "PASS"
assert pos_cash_features["pos_dpd_disagreement_months"].sum() == mapped_contracts["pos_dpd_disagreement_months"].sum()
assert pos_cash_features["pos_remaining_increase_contracts"].sum() == mapped_contracts["pos_has_remaining_increase"].sum()
assert pos_cash_features["pos_contracts_with_history_gaps"].sum() == mapped_contracts["pos_has_history_gaps"].sum()
assert pos_cash_features["pos_gap_months_total"].sum() == mapped_contracts["pos_gap_months"].sum()


### Check Applicant Features Directly Against Monthly Source Rows

For the retained applicant sample we fetch previous IDs from the mapping, then
inspect all their monthly records. A simple loop is useful here because the
sample is small. It independently checks monthly weighting and contract-level
event counts rather than relying only on total conservation.


In [48]:
check_features = pos_cash_features.set_index("SK_ID_CURR")
for applicant_id in check_applicant_ids:
    previous_ids = check_previous_mapping.loc[check_previous_mapping["SK_ID_CURR"].eq(applicant_id), "SK_ID_PREV"]
    records = check_applicant_rows.loc[check_applicant_rows["SK_ID_PREV"].isin(previous_ids)]
    actual = check_features.loc[applicant_id]
    assert actual["pos_total_previous_contracts"] == len(previous_ids)
    assert actual["pos_contracts_with_history"] == records["SK_ID_PREV"].nunique()
    assert actual["pos_monthly_records_total"] == len(records)
    if len(records):
        month_counts = records.groupby("SK_ID_PREV").size()
        np.testing.assert_allclose(actual["pos_history_months_mean"], month_counts.mean())
    for source, prefix in DPD_FAMILIES.items():
        values = records[source].dropna().to_numpy(dtype="float64")
        expected_mean = values.mean() if len(values) else np.nan
        expected_rate = (values > 0).mean() if len(values) else np.nan
        np.testing.assert_allclose(actual[prefix + "_mean"], expected_mean, equal_nan=True)
        np.testing.assert_allclose(actual[prefix + "_positive_rate"], expected_rate, equal_nan=True)
        np.testing.assert_allclose(actual[prefix + "_max"], values.max() if len(values) else np.nan, equal_nan=True)
        positives = records.loc[records[source].gt(0).fillna(False)]
        np.testing.assert_allclose(actual[prefix + "_delinquent_contracts"], positives["SK_ID_PREV"].nunique() if len(values) else np.nan, equal_nan=True)
        repeated_counts = positives.groupby("SK_ID_PREV").size()
        np.testing.assert_allclose(actual[prefix + "_repeated_contracts"], repeated_counts.ge(2).sum() if len(values) else np.nan, equal_nan=True)
        recent_records = records.loc[records["MONTHS_BALANCE"].between(RECENT_MIN_MONTH, RECENT_MAX_MONTH)]
        recent_days = recent_records[source].dropna().to_numpy(dtype="float64")
        np.testing.assert_allclose(actual[prefix + "_recent_6m_positive_rate"], (recent_days > 0).mean() if len(recent_days) else np.nan, equal_nan=True)
        latest_positive_month = positives["MONTHS_BALANCE"].max()
        np.testing.assert_allclose(actual[prefix + "_last_positive_month"], latest_positive_month, equal_nan=True)
validation_results["Direct monthly-source applicant checks, including weighted means and recency"] = "PASS"
print("Applicants checked directly:", len(check_applicant_ids))
del check_features, check_applicant_rows, check_previous_mapping


Applicants checked directly: 43


### Rate Formulas and Numeric Limits

Every rate uses its documented denominator. Replacing a zero denominator by
missing avoids infinities and avoids pretending that no observed history has a
zero delinquency rate. Binary recent indicators follow the same availability rule.


In [49]:
rate_definitions = {
    "pos_history_coverage_ratio": ("pos_contracts_with_history", "pos_total_previous_contracts"),
    "pos_active_contract_ratio": ("pos_latest_active_contracts", "pos_contracts_with_history"),
    "pos_completed_contract_ratio": ("pos_latest_completed_contracts", "pos_contracts_with_history"),
    "pos_dpd_disagreement_rate": ("pos_dpd_disagreement_months", "pos_dpd_comparable_months")
}
for prefix in DPD_FAMILIES.values():
    rate_definitions[prefix + "_positive_rate"] = (prefix + "_positive_months", prefix + "_observed_months")
    rate_definitions[prefix + "_contract_delinquency_ratio"] = (prefix + "_delinquent_contracts", prefix + "_observed_contracts")
    rate_definitions[prefix + "_recent_6m_positive_rate"] = (prefix + "_recent_6m_positive_months", prefix + "_recent_6m_observed_months")
assert set(rate_definitions) == set(RATE_COLUMNS)
for feature, (numerator, denominator) in rate_definitions.items():
    expected = pos_cash_features[numerator] / pos_cash_features[denominator].replace(0, np.nan)
    np.testing.assert_allclose(pos_cash_features[feature].to_numpy(dtype="float64", na_value=np.nan), expected.to_numpy(dtype="float64", na_value=np.nan), equal_nan=True)
    assert pos_cash_features[feature].dropna().between(0, 1).all(), feature
    assert pos_cash_features[feature].isna().eq(pos_cash_features[denominator].eq(0)).all(), feature
assert not np.isinf(pos_cash_features.to_numpy(dtype="float64", na_value=np.nan)).any()
validation_results["Documented rate formulas, 0–1 ranges, and no infinities"] = "PASS"


### Inspect and Explain Missingness

- No mapped POS history gives zero availability counts and missing history statistics.
- No known DPD gives missing DPD counts/mean/max/rates, with a zero observed-value count.
- No evaluable adjacent pair gives missing consecutive or installment-increase counts.
- No observed recent DPD gives missing recent event measurements and rates.
- No positive DPD gives a missing last-positive offset, even when all known DPD is zero.
- Missing latest installment statistics mean the final snapshots supply no usable value;
  we never take an older nonmissing value to replace a missing latest value.

القيمة الناقصة معناها إن المعلومات اللازمة للحساب مش متاحة. الصفر مع بيانات معروفة
معناه إننا لم نرصد الحالة دي في السجل المتاح، مش ضمان عدم حدوثها طول عمر القرض.


In [50]:
missing_summary = pd.DataFrame({
    "missing_applicants": pos_cash_features.isna().sum(),
    "missing_percent": (pos_cash_features.isna().mean() * 100).round(2)
}).sort_values("missing_applicants", ascending=False)
display(missing_summary.loc[missing_summary["missing_applicants"].gt(0)])
assert pos_cash_features[COMPLETE_COUNT_COLUMNS].notna().all().all()
history_stat_columns = [column for column in pos_history_features if column not in COMPLETE_COUNT_COLUMNS + ["SK_ID_CURR"]]
for column in history_stat_columns:
    assert pos_cash_features[column].isna().eq(no_pos_history).all(), column
for prefix in DPD_FAMILIES.values():
    no_known_dpd = pos_cash_features[prefix + "_observed_months"].eq(0)
    for suffix in ["_mean", "_max", "_positive_months", "_delinquent_contracts", "_repeated_contracts", "_ge90_contracts"]:
        assert pos_cash_features[prefix + suffix].isna().eq(no_known_dpd).all(), prefix + suffix
    assert pos_cash_features[prefix + "_consecutive_contracts"].isna().eq(pos_cash_features[prefix + "_consecutive_evaluable_contracts"].eq(0)).all()
    assert pos_cash_features[prefix + "_last_positive_month"].isna().eq(pos_cash_features[prefix + "_positive_months"].fillna(0).eq(0)).all()
    no_known_recent = pos_cash_features[prefix + "_recent_6m_observed_months"].eq(0)
    assert pos_cash_features[prefix + "_recent_6m_positive_months"].isna().eq(no_known_recent).all()
    assert pos_cash_features[prefix + "_has_recent_6m_delinquency"].isna().eq(no_known_recent).all()
    assert pos_cash_features[prefix + "_has_recent_6m_delinquency"].dropna().isin([0, 1]).all()


,missing_applicants,missing_percent
pos_dpd_def_last_positive_month,294887,87.02
pos_dpd_last_positive_month,279610,82.52
pos_dpd_has_recent_6m_delinquency,147135,43.42
pos_dpd_def_has_recent_6m_delinquency,147135,43.42
pos_dpd_def_recent_6m_positive_rate,147135,43.42
pos_dpd_recent_6m_positive_months,147135,43.42
pos_dpd_recent_6m_positive_rate,147135,43.42
pos_dpd_def_recent_6m_positive_months,147135,43.42
pos_remaining_increase_contracts,4828,1.42
pos_dpd_consecutive_contracts,4801,1.42


In [51]:
installment_null_rules = {
    "pos_latest_total_instalments_mean": (pos_cash_features["pos_contracts_with_history"] - pos_cash_features["pos_latest_missing_total_instalment_contracts"]).eq(0),
    "pos_latest_total_instalments_max": (pos_cash_features["pos_contracts_with_history"] - pos_cash_features["pos_latest_missing_total_instalment_contracts"]).eq(0),
    "pos_latest_remaining_instalments_mean": pos_cash_features["pos_latest_remaining_observed_contracts"].eq(0),
    "pos_latest_remaining_instalments_max": pos_cash_features["pos_latest_remaining_observed_contracts"].eq(0),
    "pos_latest_zero_remaining_contracts": pos_cash_features["pos_latest_remaining_observed_contracts"].eq(0),
    "pos_remaining_instalments_contract_mean": pos_cash_features["pos_remaining_instalments_observed_contracts"].eq(0),
    "pos_remaining_instalments_max": pos_cash_features["pos_remaining_instalments_observed_contracts"].eq(0),
    "pos_changing_total_instalment_contracts": pos_cash_features["pos_total_instalments_observed_contracts"].eq(0),
    "pos_remaining_increase_contracts": pos_cash_features["pos_remaining_increase_evaluable_contracts"].eq(0)
}
for feature, expected_nulls in installment_null_rules.items():
    assert pos_cash_features[feature].isna().eq(expected_nulls).all(), feature
assert (pos_cash_features["pos_remaining_increase_contracts"].isna() | pos_cash_features["pos_remaining_increase_contracts"].le(pos_cash_features["pos_remaining_increase_evaluable_contracts"])).all()
assert (pos_cash_features["pos_latest_zero_remaining_contracts"].isna() | pos_cash_features["pos_latest_zero_remaining_contracts"].le(pos_cash_features["pos_latest_remaining_observed_contracts"])).all()
assert (pos_cash_features["pos_changing_total_instalment_contracts"].isna() | pos_cash_features["pos_changing_total_instalment_contracts"].le(pos_cash_features["pos_total_instalments_observed_contracts"])).all()
for feature in ["pos_dpd_disagreement_months", "pos_dpd_disagreement_contracts"]:
    assert pos_cash_features[feature].isna().eq(pos_cash_features["pos_dpd_comparable_months"].eq(0)).all()
validation_results["Feature nulls agree with history, DPD, recent-window, and installment availability"] = "PASS"


## 10. Save the Applicant Features to Parquet

The table contains `SK_ID_CURR` plus numeric `pos_` features, with no saved index.
We use the same `data/interim/` convention as the existing feature notebooks.

To preserve existing outputs, we create a new file exclusively. On a repeat run,
we verify an existing POS feature file is exactly equal and leave it unchanged.
If it differs, execution stops for review instead of overwriting it.


In [52]:
DATA_INTERIM.mkdir(parents=True, exist_ok=True)
if OUTPUT_PATH.exists():
    existing_pos_features = pd.read_parquet(OUTPUT_PATH)
    pd.testing.assert_frame_equal(existing_pos_features, pos_cash_features, check_exact=True)
    print("Existing identical output verified and preserved:", OUTPUT_PATH)
    del existing_pos_features
else:
    with OUTPUT_PATH.open("xb") as output_file:
        pos_cash_features.to_parquet(output_file, index=False)
    print("Saved:", OUTPUT_PATH)


Saved: e:\Depi Notes Data Engineering\DEPI project R5\ProJect\data\interim\pos_cash_features.parquet


## 11. Reload and Verify the Output

A successful read is not enough. We compare shape, unique IDs, column order,
dtypes, exact values, and nulls with the in-memory table. We also fingerprint
both raw CSVs again after the complete workflow.


In [53]:
saved_pos_cash = pd.read_parquet(OUTPUT_PATH)
print("Saved shape:", saved_pos_cash.shape)
print("Unique IDs:", saved_pos_cash["SK_ID_CURR"].is_unique)
print("Duplicate IDs:", saved_pos_cash["SK_ID_CURR"].duplicated().sum())
assert saved_pos_cash.shape == pos_cash_features.shape
assert saved_pos_cash["SK_ID_CURR"].is_unique
assert saved_pos_cash["SK_ID_CURR"].notna().all()
pd.testing.assert_frame_equal(saved_pos_cash, pos_cash_features, check_exact=True)
pd.testing.assert_series_equal(saved_pos_cash.isna().sum(), pos_cash_features.isna().sum())
validation_results["Parquet reload exactly preserves shape, IDs, dtypes, values, and nulls"] = "PASS"
saved_pos_cash.head()


Saved shape: (338857, 89)
Unique IDs: True
Duplicate IDs: 0


,SK_ID_CURR,pos_total_previous_contracts,pos_contracts_with_history,pos_latest_active_contracts,pos_latest_completed_contracts,pos_latest_demand_contracts,pos_latest_amortized_debt_contracts,pos_latest_other_status_contracts,pos_latest_unknown_status_contracts,pos_previous_contracts_without_pos_history,...,pos_remaining_increase_evaluable_contracts,pos_dpd_comparable_months,pos_dpd_disagreement_months,pos_dpd_disagreement_contracts,pos_remaining_exceeds_total_contracts,pos_latest_completed_with_remaining_contracts,pos_latest_active_zero_remaining_contracts,pos_dpd_latest_zero_remaining_positive_contracts,pos_dpd_def_latest_zero_remaining_positive_contracts,pos_dpd_disagreement_rate
0,100001,1,1,0,1,0,0,0,0,0,...,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,100002,1,1,1,0,0,0,0,0,0,...,1,19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,100003,3,3,1,2,0,0,0,0,0,...,3,28,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,100004,1,1,0,1,0,0,0,0,0,...,1,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,100005,2,1,0,1,0,0,0,0,1,...,1,11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [54]:
for path, original_sha256 in raw_sha256_before.items():
    with path.open("rb") as raw_file:
        current_sha256 = hashlib.file_digest(raw_file, "sha256").hexdigest()
    assert current_sha256 == original_sha256, f"Raw input changed: {path.name}"
validation_results["Both raw CSV SHA-256 fingerprints unchanged"] = "PASS"
display(saved_pos_cash.dtypes.value_counts().rename("columns").to_frame())
display(pd.Series(validation_results, name="Result").to_frame())


,columns
int32,36
float64,27
float32,26


,Result
"Source schema, IDs, numeric domains, and pre-application offsets",PASS
Exact duplicates reported; analytical contract-month keys unique,PASS
"Unambiguous mapping, source-ID audit, and matched/unmatched conservation",PASS
"One row per contract, source month totals, latest full snapshots, and independent gaps",PASS
"SK_DPD monthly totals, severity, consecutive counts, means, and rate bounds",PASS
"SK_DPD_DEF monthly totals, severity, consecutive counts, means, and rate bounds",PASS
Random and boundary contract checks directly against monthly DPD and offsets,PASS
"Latest installments, changing terms, evaluable increases, and missingness",PASS
"Unique applicant grain, exact mapping population, and contract/month conservation",PASS
Every applicant latest-status count verified independently,PASS


## 12. Final Summary

This report uses values calculated during this execution. Source applicants
are an optional CSV audit; mapped POS applicants and final mapping applicants
are different populations and are reported separately.


In [55]:
n_engineered_features = saved_pos_cash.shape[1] - 1
category_summary = "\n".join(f"| {name} | {group.shape[1] - 1} |" for name, group in feature_groups.items())
validation_summary = "\n".join(f"| {check} | {result} |" for check, result in validation_results.items())
display(Markdown(f"""
| Summary | Result |
|---|---|
| Source dataset dimensions | {source_shape[0]:,} monthly rows × {source_shape[1]} columns |
| Analytical rows after exact-duplicate handling | {analytical_shape[0]:,} |
| Unique POS/CASH contracts | {n_source_contracts:,} |
| Mapped / unmatched POS contracts | {len(matched_ids):,} / {len(unmatched_ids):,} |
| Contract match coverage | {len(matched_ids) / n_source_contracts:.2%} |
| Unmatched monthly rows retained at contract grain | {n_unmatched_monthly_rows:,} |
| Source observed applicants (audit only) | {f'{n_source_applicants:,}' if n_source_applicants is not None else 'Column absent'} |
| Applicants with mapped POS history | {n_mapped_pos_applicants:,} |
| Unique applicants in final mapping population | {n_applicants:,} |
| Applicants with no mapped POS history | {int(no_pos_history.sum()):,} |
| Engineered features, excluding SK_ID_CURR | {n_engineered_features} |
| Final and reloaded output shape | {saved_pos_cash.shape} |
| Contracts with history gaps (all POS source contracts) | {int(contract_features['pos_has_history_gaps'].sum()):,} |
| Median distinct observed months per source contract | {contract_features['pos_observed_months'].median():.0f} |
| Source contracts with changing total terms | {int(contract_features['pos_has_changing_total_instalments'].sum()):,} |
| Source contracts with remaining-count increases | {int(contract_features['pos_has_remaining_increase'].sum()):,} |
| Latest Active / Completed source contracts | {int(contract_features['pos_is_latest_active'].sum()):,} / {int(contract_features['pos_is_latest_completed'].sum()):,} |
| Source ordinary / adjusted positive DPD rows | {source_dpd_audit['pos_dpd']['positive_months']:,} / {source_dpd_audit['pos_dpd_def']['positive_months']:,} |
| Source DPD disagreement / adjusted-greater rows | {dpd_disagreement_rows:,} / {adjusted_greater_rows:,} |
| Exact duplicate extras / unresolved duplicate-key rows | {exact_duplicate_extras:,} / {conflicting_key_rows:,} |
| Output file | `{OUTPUT_PATH.relative_to(REPO_ROOT)}` |
| Raw input contents unchanged | Both SHA-256 fingerprints matched |

| Feature category | Features |
|---|---|
{category_summary}

| Validation | Result |
|---|---|
{validation_summary}
"""))
assert sum(group.shape[1] - 1 for group in feature_groups.values()) == n_engineered_features



| Summary | Result |
|---|---|
| Source dataset dimensions | 10,001,358 monthly rows × 8 columns |
| Analytical rows after exact-duplicate handling | 10,001,358 |
| Unique POS/CASH contracts | 936,325 |
| Mapped / unmatched POS contracts | 898,903 / 37,422 |
| Contract match coverage | 96.00% |
| Unmatched monthly rows retained at contract grain | 340,561 |
| Source observed applicants (audit only) | 337,252 |
| Applicants with mapped POS history | 334,359 |
| Unique applicants in final mapping population | 338,857 |
| Applicants with no mapped POS history | 4,498 |
| Engineered features, excluding SK_ID_CURR | 88 |
| Final and reloaded output shape | (338857, 89) |
| Contracts with history gaps (all POS source contracts) | 1,950 |
| Median distinct observed months per source contract | 10 |
| Source contracts with changing total terms | 304,293 |
| Source contracts with remaining-count increases | 2,651 |
| Latest Active / Completed source contracts | 236,149 / 698,421 |
| Source ordinary / adjusted positive DPD rows | 295,227 / 113,969 |
| Source DPD disagreement / adjusted-greater rows | 183,880 / 0 |
| Exact duplicate extras / unresolved duplicate-key rows | 0 / 0 |
| Output file | `data\interim\pos_cash_features.parquet` |
| Raw input contents unchanged | Both SHA-256 fingerprints matched |

| Feature category | Features |
|---|---|
| Contract counts, latest status, and coverage | 12 |
| History length and gaps | 10 |
| Ordinary DPD frequency and severity | 14 |
| Adjusted DPD frequency and severity | 14 |
| Recent relative-window delinquency | 12 |
| Installment progress and availability | 17 |
| Quality and DPD differences | 9 |

| Validation | Result |
|---|---|
| Source schema, IDs, numeric domains, and pre-application offsets | PASS |
| Exact duplicates reported; analytical contract-month keys unique | PASS |
| Unambiguous mapping, source-ID audit, and matched/unmatched conservation | PASS |
| One row per contract, source month totals, latest full snapshots, and independent gaps | PASS |
| SK_DPD monthly totals, severity, consecutive counts, means, and rate bounds | PASS |
| SK_DPD_DEF monthly totals, severity, consecutive counts, means, and rate bounds | PASS |
| Random and boundary contract checks directly against monthly DPD and offsets | PASS |
| Latest installments, changing terms, evaluable increases, and missingness | PASS |
| Unique applicant grain, exact mapping population, and contract/month conservation | PASS |
| Every applicant latest-status count verified independently | PASS |
| SK_DPD applicant counts conserved from mapped contracts | PASS |
| SK_DPD_DEF applicant counts conserved from mapped contracts | PASS |
| Direct monthly-source applicant checks, including weighted means and recency | PASS |
| Documented rate formulas, 0–1 ranges, and no infinities | PASS |
| Feature nulls agree with history, DPD, recent-window, and installment availability | PASS |
| Parquet reload exactly preserves shape, IDs, dtypes, values, and nulls | PASS |
| Both raw CSV SHA-256 fingerprints unchanged | PASS |


### Decisions and Limitations

- **Mapping and population:** The final table covers all applicants in the
  previous-application ID mapping. Only matched POS contracts contribute monthly
  measurements. Unmatched contracts and their counts remain inspectable in
  `unmatched_pos_contracts`; their optional source applicant IDs do not bypass
  the mapping. Applicants absent from the previous mapping are handled later
  when feature tables are joined to the main applicant table.
- **Without history:** Counts without POS history refer to all previous IDs.
  The ID-only mapping cannot prove these are POS loans expected to have history.
  Product-specific coverage would need a separately justified product definition.
- **Observed lifecycle:** Latest status is the latest observed snapshot, not
  necessarily today's or the application-time status. Histories can be censored
  or have gaps. Monthly-record counts and time spans remain distinct.
- **Delinquency:** Ordinary and tolerance-adjusted DPD stay separate. Positive
  months are observations, not necessarily separate late-payment events.
  Repeated, consecutive, and ≥90-day flags do not establish default. Longer
  observation increases detection opportunity; no predictive claim is made here.
- **Recent window:** `-6` through `-1` is a reviewable six-offset choice. No
  calendar dates are inferred, and no missing window is called a clean window.
- **Installments and quality:** Changing terms and remaining increases are
  retained rather than automatically corrected. Adjacent-snapshot increases
  can cross gaps. Completed/remaining combinations and remaining > total are
  review flags. Zero remaining does not prove all arrears are settled. Exact
  payment and tolerance rules would require additional source context.
- **Missingness and weighting:** Only explicit availability/status counts use
  zero for empty histories. DPD means/rates are monthly-observation weighted;
  history-length and historical remaining-count means weight contracts equally.
  Latest-state means use only the final contract snapshots, including their missingness.
- **Leakage and reproducibility:** All used monthly offsets precede application
  time. No target or target-derived feature is loaded. Production use still
  requires checking source availability as of scoring time. Exact duplicates
  can be removed with counts reported; conflicting keys or mappings stop the
  workflow. The raw inputs and existing outputs are preserved.
